# GLM-5.3-Flash on 4x CMP 170HX (SM80) — 4card-pp4, vLLM

| Metric | Value |
|---|---|
| Decode, c=1 | **87.6 tok/s** (P1 math, temp 0, median of 3, clean) / **67.9 tok/s** (P2, temp 0.7 + `ignore_eos`, median of 5) |
| Best aggregate | 78.4 tok/s at c=16 (4k prompt, 256 out) — *measured on degraded link* |
| Prefill | 1,752 tok/s at 16,384 tokens (median of 3) — *measured on degraded link* |
| TTFT | 9.44 s at 16,384 tokens / 3.67 s at 4,096 tokens (warm, streaming) — *measured on degraded link* |

![context sweep](../assets/charts/2026-09-05-glm-5.3-flash-pp4-context-sweep.png)

```bash
docker pull ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905
```

Guide page: [docs/models/glm-5.3-flash.md](../docs/models/glm-5.3-flash.md)

<!-- video cell -->
**Result video:** not yet rendered. A ten-second cut belongs here once the lane's
quality and stability cells land; see `assets/video/` for the pattern used by the
DeepSeek-V4-Flash-Vision-Exp notebooks.

In [1]:
# --- Status cell ---
EXPERIMENT = "2026-09-05-glm-5.3-flash-4card-pp4-vllm"
RESULTS_DIR = "../results/" + EXPERIMENT
RECEIPTS = RESULTS_DIR + "/receipts"
LIVE = False

print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : measured (PP4 + MTP k=3 recipe of record; depth sweep, context")
print("             sweep, concurrency, prefill/TTFT filled. Lossless, stability,")
print("             quality, and the NVFP4 cells are untested (pending).)")

experiment : 2026-09-05-glm-5.3-flash-4card-pp4-vllm
LIVE       : False
status     : measured (PP4 + MTP k=3 recipe of record; depth sweep, context
             sweep, concurrency, prefill/TTFT filled. Lossless, stability,
             quality, and the NVFP4 cells are untested (pending).)


In [2]:
# --- Helpers: receipt loader and table renderer ---
import json, os, statistics as st
from IPython.display import display, Markdown


def receipt(*parts):
    """Load one committed receipt. LIVE=True would re-measure; this notebook replays."""
    path = os.path.join(RECEIPTS, *parts)
    with open(path) as fh:
        return json.load(fh)


def render_table(headers, rows):
    lines = ["| " + " | ".join(str(h) for h in headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))


def fmt(v, nd=2):
    return "untested (pending)" if v is None else f"{v:,.{nd}f}"


assert not LIVE, "commit this notebook with LIVE = False"
print("receipts:", len(os.listdir(RECEIPTS)), "entries under", RECEIPTS)

receipts: 10 entries under ../results/2026-09-05-glm-5.3-flash-4card-pp4-vllm/receipts


## 1. TL;DR

Pipeline parallelism, not tensor parallelism, is the right topology for
GLM-5.3-Flash on this four-card node. PP4 with the model's native MTP drafter at
depth 3 is now the **recipe of record**; the 2026-09-03 TP4 result (60.4 tok/s
c=1 median) is **superseded** and kept in the appendix, because TP4 is link-bound
on a fabric with no NVLink and no P2P — the same TP4 recipe measured 70.5 tok/s
before one card's PCIe link retrained narrow and 14.4 tok/s after, while PP4 on
the same degraded box holds 60.8 tok/s under the identical protocol.

The lane derives from a community pipeline-parallel patch set; see
**Attribution** in section 3.

In [3]:
pins = {
    "model": "GLM-5.3-Flash",
    "checkpoint": "wtdcode/GLM-5.3-Flash-AWQ-W4A16",
    "checkpoint_revision": "abd7b07719111f137e1de8a0c1b7e01c11b74d1a",
    "checkpoint_bytes": 190843146533,
    "quantization": "AWQ W4A16 (compressed-tensors)",
    "runtime_source": "PixelML/sm80vllm @ pp-dflash2/glm53-flash-487ecf187-20260905",
    "runtime_base": "vllm/vllm-openai:glm53-flash @ 487ecf187 + 24 ported patches",
    "image": "ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905",
    "image_index_digest": "sha256:62f612b49614523e6a46e1493d35d3efd1f363917129d38cc923a31053693bfb",
    "recipe_script": "recipes/glm53-flash-4x170hx-pp4.sh",
    "topology": "PP4, VLLM_PP_LAYER_PARTITION=14,12,12,7 (45 hidden layers)",
    "speculation": "native MTP, num_speculative_tokens=3",
    "max_model_len": 393216,
    "prefix_caching": "off (--no-enable-prefix-caching)",
    "gpu_memory_utilization": 0.90,
    "max_num_seqs": 8,
    "max_num_batched_tokens": 4096,
    "micro_batch_cap": "VLLM_PP_MAX_DECODE_REQS_PER_BATCH=2",
    "sidecar_block_size": "VLLM_GLM5N_SIDECAR_BLOCK_SIZE=256",
    "kv_dtype": "auto",
    "cards": "4x CMP 170HX (SM80, 64 GiB each), 180 W per-card cap, no NVLink",
    "link_state": "degraded: GPU1 PCIe Gen1 x1 (slot ceiling x8), GPU0 x8, GPU2/3 x16",
    "measured_utc": "2026-09-05",
}
for k, v in pins.items():
    print(f"{k:24s} {v}")

model                    GLM-5.3-Flash
checkpoint               wtdcode/GLM-5.3-Flash-AWQ-W4A16
checkpoint_revision      abd7b07719111f137e1de8a0c1b7e01c11b74d1a
checkpoint_bytes         190843146533
quantization             AWQ W4A16 (compressed-tensors)
runtime_source           PixelML/sm80vllm @ pp-dflash2/glm53-flash-487ecf187-20260905
runtime_base             vllm/vllm-openai:glm53-flash @ 487ecf187 + 24 ported patches
image                    ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905
image_index_digest       sha256:62f612b49614523e6a46e1493d35d3efd1f363917129d38cc923a31053693bfb
recipe_script            recipes/glm53-flash-4x170hx-pp4.sh
topology                 PP4, VLLM_PP_LAYER_PARTITION=14,12,12,7 (45 hidden layers)
speculation              native MTP, num_speculative_tokens=3
max_model_len            393216
prefix_caching           off (--no-enable-prefix-caching)
gpu_memory_utilization   0.9
max_num_seqs             8
max_num_batched_tokens   4096
micro_batch_c

### Protocol

Two protocols run side by side so the lane is comparable both to the upstream
patch author's numbers and to this club's own history.

| | P1 | P2 |
|---|---|---|
| Source | `promisezackr/glm53-flash-170hx-pp8` `scripts/bench.py` | this club's `bench_glm53.py`, used for the 2026-09-03 TP4 record |
| Sampling | temperature 0 (greedy) | temperature 0.7, `ignore_eos` |
| Output tokens | 512 | 512 |
| Repetitions | 3, median reported | 5, median reported (first rep is cold) |
| Workloads | code, json, counting, math, prose (+ a repetition diagnostic) | one fixed long-form prompt |
| Degeneracy guard | the author's repeat guard flags a repetition-collapsed completion; flagged cells are inflated and are **not** read as throughput | none — read beside P1 |

Token counts come from the final `usage` object of each response, never from
counting stream events.

**Headline rule:** the c=1 headline is the best P1 workload whose completion is
clean, with the P2 median printed beside it.

## 2. Visible results

### 2.1 Boot reliability and load gate

In [4]:
gate = receipt("k3", "gate.json")
cells = receipt("cells.json")
sanity = cells["sanity"]

render_table(
    ["Check", "Result"],
    [
        ["Boots served / attempted, this recipe", "**8 / 8** (795-1,263 s to `Application startup complete`), plus 2 / 2 on the earlier port run"],
        ["Boot time, the measured boot", "1,029 s (engine init incl. profile + KV + warmup: 320.4 s)"],
        ["`/v1/models` reachable", "yes"],
        ["Deterministic greedy repeat (3x, 1 token)", f'{gate["verdict"]} — identical: {gate["deterministic_identical"]}'],
        ["Greedy sanity prompts (64 tokens each)", f'{sum(1 for s in sanity if s.get("clean"))} / {len(sanity)} clean and factually correct'],
        ["KV pool at max-model-len 393,216", "1,194,627 tokens (3.04x a single max-length request)"],
        ["GPU health before and after", "4/4 `rev a1`, zero Xid, zero ECC"],
        ["Fastest boot", "795 s (the speculation-off boot); 873 s was the post-recovery boot with a warm compile cache"],
        ["Idle memory per card", "51.6-54.2 GiB of 64 GiB"],
        ["Power cap", "180 W verified on all four cards throughout"],
        ["Peak temperature under load", "53 C (sampled after each stability round, section 2.7)"],
    ])

| Check | Result |
|---|---|
| Boots served / attempted, this recipe | **8 / 8** (795-1,263 s to `Application startup complete`), plus 2 / 2 on the earlier port run |
| Boot time, the measured boot | 1,029 s (engine init incl. profile + KV + warmup: 320.4 s) |
| `/v1/models` reachable | yes |
| Deterministic greedy repeat (3x, 1 token) | PASS — identical: True |
| Greedy sanity prompts (64 tokens each) | 3 / 3 clean and factually correct |
| KV pool at max-model-len 393,216 | 1,194,627 tokens (3.04x a single max-length request) |
| GPU health before and after | 4/4 `rev a1`, zero Xid, zero ECC |
| Fastest boot | 795 s (the speculation-off boot); 873 s was the post-recovery boot with a warm compile cache |
| Idle memory per card | 51.6-54.2 GiB of 64 GiB |
| Power cap | 180 W verified on all four cards throughout |
| Peak temperature under load | 53 C (sampled after each stability round, section 2.7) |

### 2.2 MTP draft depth: per-workload c=1 decode (P1)

One boot per depth, everything else identical. `(deg)` marks a cell the author's
repeat guard flagged as repetition-collapsed; those cells are inflated and must
not be read as throughput. c=1 is link-insensitive under PP4, so this table is
**not** degraded-link limited.

In [5]:
KS = ["nospec", "k2", "k3", "k5", "k7"]
WORKLOADS = ["counting", "json", "code", "math", "prose", "repetition"]
p1 = {k: receipt(k, "p1.json") for k in KS}

rows = []
for w in WORKLOADS:
    row = [w + (" *(diagnostic)*" if w == "repetition" else "")]
    for k in KS:
        wl = p1[k]["workloads"][w]
        deg = any(r.get("degenerate") for r in wl["reps"])
        row.append(f'{wl["median_tok_s"]:.2f}' + (" *(deg)*" if deg else ""))
    rows.append(row)

headline = []
for k in KS:
    best, best_v = None, -1
    for w in WORKLOADS:
        if w == "repetition":
            continue
        wl = p1[k]["workloads"][w]
        if any(r.get("degenerate") for r in wl["reps"]):
            continue
        if wl["median_tok_s"] > best_v:
            best, best_v = w, wl["median_tok_s"]
    headline.append(f"**{best_v:.2f}** {best}")
rows.append(["**P1 headline, clean cells only**"] + headline)

render_table(["workload (median of 3, tok/s)", "k=0 (off)", "k=2", "k=3", "k=5", "k=7"], rows)

| workload (median of 3, tok/s) | k=0 (off) | k=2 | k=3 | k=5 | k=7 |
|---|---|---|---|---|---|
| counting | 42.88 | 81.89 | 75.69 | 63.73 | 53.40 |
| json | 42.17 *(deg)* | 84.08 *(deg)* | 87.02 | 82.10 | 78.16 *(deg)* |
| code | 42.83 | 60.52 | 58.37 *(deg)* | 40.65 | 34.64 |
| math | 42.94 | 80.88 | 87.55 | 72.52 | 91.44 |
| prose | 42.91 | 60.73 | 60.54 | 45.74 | 47.30 |
| repetition *(diagnostic)* | 42.88 *(deg)* | 74.32 | 80.21 | 77.36 | 78.42 *(deg)* |
| **P1 headline, clean cells only** | **42.94** math | **81.89** counting | **87.55** math | **82.10** json | **91.44** math |

### 2.3 MTP draft depth: acceptance and P2 decode

Acceptance is read from the engine's own `vllm:spec_decode_*` counters,
differenced across the measurement window — an exact count, not a gauge. The
k=3 pair is stored at the receipt root; the other depths are stored under their
own directory.

The k-sweep reading: per-verified-token cost on this box is **not** flat, so
deeper drafts do not pay for themselves. k=3 is the record. Acceptance, not
depth, is the lever — k=7 buys a longer accepted run (3.61 vs 2.96) but at a
draft-acceptance rate of 37.3% vs 65.4%, and the extra sequential MTP forwards
cost more than the extra accepted tokens return on every workload except math.
k=7 wins math alone; it loses code (34.64 vs 58.37), counting (53.40 vs 75.69)
and prose (47.30 vs 60.54), and its P2 median is 38.13 tok/s against k=3's
67.91. k=2 is the closest rival and loses on P2 as well.

In [6]:
def accept(*parts):
    b = receipt(*(parts + ("accept_before.json",)))
    a = receipt(*(parts + ("accept_after.json",)))
    pick = lambda d, s: next(v for kk, v in d.items() if s in kk)
    drafts = pick(a, "num_drafts") - pick(b, "num_drafts")
    dtok = pick(a, "num_draft_tokens") - pick(b, "num_draft_tokens")
    acc = pick(a, "num_accepted") - pick(b, "num_accepted")
    return drafts, dtok, acc, acc / dtok, acc / drafts + 1

ACC_DIR = {"k2": ("k2",), "k3": (), "k5": ("k5",), "k7": ("k7",)}
rows = []
for k in KS:
    dc = receipt(k, "decode_c1.json")
    reps = [r["decode_tok_s"] for r in dc["reps"]]
    if k == "nospec":
        cols = ["k=0 (off)", "—", "—", "—", "no drafter", "1.00"]
    else:
        drafts, dtok, acc, rate, alen = accept(*ACC_DIR[k])
        cols = [k.replace("k", "k="), f"{int(drafts):,}", f"{int(dtok):,}",
                f"{int(acc):,}", f"{rate * 100:.1f}%", f"{alen:.2f}"]
    rows.append(cols + [f"{st.median(reps):.2f}", f"{max(reps):.2f}", f"{reps[0]:.2f}"])

render_table(
    ["depth", "drafts", "draft tokens", "accepted",
     "draft acceptance rate", "mean accepted length",
     "P2 median tok/s (5 reps)", "P2 peak", "P2 cold rep"],
    rows)

| depth | drafts | draft tokens | accepted | draft acceptance rate | mean accepted length | P2 median tok/s (5 reps) | P2 peak | P2 cold rep |
|---|---|---|---|---|---|---|---|---|
| k=0 (off) | — | — | — | no drafter | 1.00 | 41.95 | 42.09 | 41.71 |
| k=2 | 7,352 | 14,704 | 10,885 | 74.0% | 2.48 | 56.54 | 66.53 | 51.95 |
| k=3 | 3,921 | 11,763 | 7,690 | 65.4% | 2.96 | 67.91 | 92.58 | 56.32 |
| k=5 | 5,422 | 27,110 | 12,641 | 46.6% | 3.33 | 51.26 | 53.56 | 52.13 |
| k=7 | 5,106 | 35,742 | 13,321 | 37.3% | 3.61 | 38.13 | 58.81 | 36.67 |

### 2.4 Decode vs context, c=1 (P4 context sweep)

One boot, the two thinking arms alternated per length, 512 output tokens,
3 repetitions plus a cold/warm pair per point. **Generation tok/s and ms/token
are link-insensitive under PP4** (one hidden-state hop of roughly 50 KB per
decode step). **Prompt processing and TTFT were measured on the degraded link
and are lower bounds.**

Warm equals cold at every length because the recipe of record runs with prefix
caching off. The two arms are statistically identical for a reason: **the
thinking switch is not switchable on this checkpoint** — the probe
(`receipts/k3/thinking_probe.json`) got no `reasoning_content` back under any
key, in either direction, including the server default. The two arms are
therefore the same configuration measured twice, and are reported as one curve.
Treating them as an A/B would be wrong.

In [7]:
sweep = receipt("k3", "sweep", "context_sweep.json")
rows = []
for r in sweep["results"]:
    off, on = r["arms"]["thinking_off"], r["arms"]["thinking_on"]
    if not off.get("cold", {}).get("ok"):
        rows.append([f'{r["target_prompt_tokens"]:,} (target)', "—",
                     "untested (pending)", "untested (pending)",
                     "untested (pending)", "untested (pending)",
                     "prompt calibration overshot the 393,216-token limit"])
        continue
    rows.append([
        f'{r["target_prompt_tokens"]:,}',
        f'{off["cold"]["prompt_tokens"]:,}',
        f'{off["median_prompt_tps"]:,.1f}',
        f'{off["median_gen_tps"]:.2f}',
        f'{off["median_tpot_ms"]:.2f}',
        f'{off["cold_ttft_s"]:.2f} / {off["warm_ttft_s"]:.2f}',
        f'{on["median_gen_tps"]:.2f}',
    ])

render_table(
    ["target prompt tok", "actual prompt tok",
     "prompt tok/s (degraded link)", "generation tok/s",
     "ms/token", "cold / warm TTFT s (degraded link)",
     "generation tok/s, thinking-on arm"],
    rows)
display(Markdown("![context sweep](../assets/charts/2026-09-05-glm-5.3-flash-pp4-context-sweep.png)"))

| target prompt tok | actual prompt tok | prompt tok/s (degraded link) | generation tok/s | ms/token | cold / warm TTFT s (degraded link) | generation tok/s, thinking-on arm |
|---|---|---|---|---|---|---|
| 327 | 336 | 592.2 | 98.58 | 10.14 | 0.52 / 0.52 | 98.10 |
| 860 | 888 | 912.2 | 98.74 | 10.13 | 0.98 / 0.96 | 89.53 |
| 2,000 | 2,024 | 1,027.9 | 81.51 | 12.27 | 1.98 / 1.96 | 76.96 |
| 4,000 | 3,968 | 1,108.4 | 75.34 | 13.27 | 3.57 / 3.57 | 76.20 |
| 8,000 | 8,042 | 1,462.0 | 74.85 | 13.36 | 5.50 / 5.50 | 73.44 |
| 16,000 | 16,095 | 1,744.3 | 78.89 | 12.68 | 9.24 / 9.21 | 73.53 |
| 33,000 | 32,986 | 1,912.5 | 76.61 | 13.05 | 17.26 / 17.25 | 76.78 |
| 66,000 | 66,023 | 2,007.0 | 77.44 | 12.91 | 32.90 / 32.89 | 75.96 |
| 131,000 | 131,042 | 2,038.4 | 78.56 | 12.73 | 64.32 / 64.29 | 79.16 |
| 258,000 (target) | — | untested (pending) | untested (pending) | untested (pending) | untested (pending) | prompt calibration overshot the 393,216-token limit |

![context sweep](../assets/charts/2026-09-05-glm-5.3-flash-pp4-context-sweep.png)

### 2.5 Concurrency scaling

4,096-token prompts, 256 output tokens, P2 sampling. **Every row here is
link-bound and is a lower bound on a healthy box.**

In [8]:
conc = receipt("k3", "conc_sweep.json")
render_table(
    ["concurrency", "aggregate tok/s", "per-stream median tok/s",
     "e2e p50 s", "e2e p95 s", "success rate"],
    [[r["concurrency"], f'{r["aggregate_tok_s"]:.2f}',
      f'{r["per_stream_median_tok_s"]:.2f}', f'{r["e2e_p50_s"]:.2f}',
      f'{r["e2e_p95_s"]:.2f}', f'{r["succeeded"]}/{r["attempted"]}']
     for r in conc["rows"]])
print("link state on this receipt:", conc["link_state"])

| concurrency | aggregate tok/s | per-stream median tok/s | e2e p50 s | e2e p95 s | success rate |
|---|---|---|---|---|---|
| 1 | 30.21 | 30.22 | 8.47 | 8.47 | 1/1 |
| 2 | 49.80 | 25.04 | 10.28 | 10.28 | 2/2 |
| 4 | 44.38 | 11.12 | 23.05 | 23.07 | 4/4 |
| 8 | 75.51 | 9.48 | 27.03 | 27.11 | 8/8 |
| 16 | 78.36 | 7.02 | 51.97 | 52.19 | 16/16 |

link state on this receipt: measured on degraded link: GPU1 x1 (ceiling x8), GPU0 x8, GPU2/3 x16


### 2.6 Prefill and TTFT

Uncached prefill with one output token, and warm streaming TTFT with 32 output
tokens. **Both are link-bound; read as lower bounds.**

In [9]:
rows = []
for n in ("4096", "16384"):
    pf = receipt("k3", f"prefill_{n}", "prefill.json")
    tt = receipt("k3", f"prefill_{n}", "ttft.json")
    rows.append([f"{int(n):,}", f'{pf["median_prefill_tok_s"]:,.1f}',
                 f'{pf["median_wall_s"]:.2f}', f'{tt["median_ttft_s"]:.2f}',
                 f'{len(pf["reps"])} / {len(tt["reps"])}'])

render_table(
    ["prompt tokens", "prefill tok/s (degraded link)", "prefill wall s",
     "warm streaming TTFT s (degraded link)", "reps (prefill / TTFT)"],
    rows)

| prompt tokens | prefill tok/s (degraded link) | prefill wall s | warm streaming TTFT s (degraded link) | reps (prefill / TTFT) |
|---|---|---|---|---|
| 4,096 | 1,128.5 | 3.63 | 3.67 | 3 / 3 |
| 16,384 | 1,751.8 | 9.35 | 9.44 | 3 / 3 |

### 2.7 Sustained stability, power and temperature

Three rounds of c=8 back to back, 2,900-token uncached prompts with a nonce,
256 output tokens, P2 sampling, with device health checked after each round.
Aggregate throughput is link-bound; the stability verdict does not depend on it.

In [10]:
stab = receipt("k3", "c8_stability.json")
rows, temps, powers = [], [], []
for r in stab["rounds_detail"]:
    h = r["health_after"]
    vals = [[c.strip() for c in line.split(",")] for line in h["nvidia_smi"]]
    t = [int(v[1]) for v in vals]
    w = [float(v[2].split()[0]) for v in vals]
    temps += t
    powers += w
    rows.append([r["round"], f'{r["succeeded"]}/{r["attempted"]}',
                 f'{r["wall_s"]:.1f}', f'{r["aggregate_tok_s"]:.2f}',
                 f"{min(t)}-{max(t)}", f"{min(w):.1f}-{max(w):.1f}",
                 h["dmesg_xid_lines"]])

render_table(["round", "succeeded", "wall s", "aggregate tok/s *(link-bound)*",
              "temp C after round", "power W after round", "Xid lines"], rows)

v = stab["verdict"]
render_table(["verdict", "value"],
             [["rounds passed", f'{v["rounds_passed"]} / {v["rounds"]}'],
              ["requests succeeded", "24 / 24"],
              ["any failure", v["any_failure"]],
              ["Xid seen", v["xid_seen"]],
              ["temperature range across all samples", f"{min(temps)}-{max(temps)} C"],
              ["instantaneous power range across all samples",
               f"{min(powers):.1f}-{max(powers):.1f} W against a 180.00 W enforced limit"]])

| round | succeeded | wall s | aggregate tok/s *(link-bound)* | temp C after round | power W after round | Xid lines |
|---|---|---|---|---|---|---|
| 1 | 8/8 | 39.3 | 52.11 | 48-52 | 78.3-170.2 | 0 |
| 2 | 8/8 | 23.4 | 87.50 | 50-53 | 89.2-201.0 | 0 |
| 3 | 8/8 | 22.7 | 90.26 | 50-53 | 64.8-215.6 | 0 |

| verdict | value |
|---|---|
| rounds passed | 3 / 3 |
| requests succeeded | 24 / 24 |
| any failure | False |
| Xid seen | False |
| temperature range across all samples | 48-53 C |
| instantaneous power range across all samples | 64.8-215.6 W against a 180.00 W enforced limit |

Throughput **rose** across the three rounds — round 1 carries warmup — so
there is no sustained-load degradation to report. Zero Xid lines after any
round, no card dropped, and the enforced power limit read 180.00 W on all four
cards throughout.

One honest note on the power column: the per-card *enforced limit* is 180.00 W
in every sample, but individual instantaneous readings go above it, up to
215.6 W. Read those as sampling artifacts of a point-in-time query rather than
sustained draw — this run recorded no integrated energy measurement, so
sustained power is `untested (pending)`.

### 2.8 Quality

Run at the **checkpoint's own sampling defaults** taken from its
`generation_config.json` — temperature 1.0, top_p 0.95 — not at the greedy
settings used for throughput, and with thinking left at the model default.

In [11]:
q = receipt("k3", "quality.json")
print("sampling  :", q["sampling"])
print("max_tokens:", q["max_tokens"])
print("thinking  :", q["thinking"])

render_table(
    ["bucket", "score", "metric"],
    [[b["bucket"], f'{b["correct"]}/{b["n"]} = {b["score"] * 100:.1f}%', b["metric"]]
     for b in q["buckets"].values()])

sampling  : {'temperature': 1.0, 'top_p': 0.95, 'seed': 1234, 'source': 'checkpoint generation_config.json'}
max_tokens: 3072
thinking  : enabled (model default)


| bucket | score | metric |
|---|---|---|
| gsm8k | 49/50 = 98.0% | exact match on final number |
| humaneval | 19/20 = 95.0% | pass@1 by executing the reference tests |
| structured_output | 10/10 = 100.0% | strict JSON parse + required top-level fields present |

Every failure in the battery, inspected — there were only two.

In [12]:
rows = []
for b in q["buckets"].values():
    for f in b["inspected_failures"]:
        if b["bucket"] == "gsm8k":
            what = (f'gold `{f["gold"]}`, predicted `{f["pred"]}` — the arithmetic is '
                    "right and the break-even year is miscounted by one; "
                    f'finish reason `{f["finish_reason"]}`, so not a truncation')
        else:
            what = ("the emitted solution calls a helper it never defines, so the "
                    "reference test raises `NameError` rather than failing an "
                    "assertion — an incomplete generation, not a wrong algorithm")
        rows.append([b["bucket"], f.get("task_id") or f'idx {f["idx"]}', what])
    if not b["inspected_failures"]:
        rows.append([b["bucket"], "—", "no failures to inspect"])

render_table(["bucket", "item", "inspected failure"], rows)
print()
print("BF16 reference:", q["bf16_kl_note"])

| bucket | item | inspected failure |
|---|---|---|
| gsm8k | idx 12 | gold `13`, predicted `12` — the arithmetic is right and the break-even year is miscounted by one; finish reason `stop`, so not a truncation |
| humaneval | HumanEval/10 | the emitted solution calls a helper it never defines, so the reference test raises `NameError` rather than failing an assertion — an incomplete generation, not a wrong algorithm |
| structured_output | — | no failures to inspect |


BF16 reference: KL divergence against a BF16 reference cannot be measured on this pool: the BF16 checkpoint does not fit in 4x64 GiB with this runtime, and no host in the club can load it. Compared against vendor-reported numbers only.


**No parity claim is made.** With no host in the pool able to load the BF16
checkpoint, there is no reference to diverge against, so these scores stand
beside the vendor's published numbers rather than against a full-precision
baseline measured here.

### 2.9 Lossless check — and why the usual reading of it does not apply

The intended cell was a token-for-token comparison of greedy output with
speculation on against speculation off. Running the control first changed what
that comparison can mean.

**Greedy output is not reproducible on this stack, with or without
speculation.** The same 20 fixed prompts, same server, same boot, same settings,
run twice, disagree with themselves — and they disagree with speculation
*turned off* too.

In [13]:
sc_on = receipt("lossless", "self_consistency.json")
sc_off = receipt("lossless", "self_consistency_nospec.json")
on_off = receipt("lossless", "lossless_on_vs_off.json")

render_table(
    ["comparison", "identical completions", "token-for-token match rate"],
    [["speculation on vs itself (noise floor)",
      f'{sc_on["identical_outputs"]}/{sc_on["n_prompts"]} = {sc_on["identical_rate"] * 100:.0f}%',
      f'{sc_on["token_for_token_match_rate"] * 100:.2f}%'],
     ["speculation **off** vs itself (noise floor)",
      f'{sc_off["identical_outputs"]}/{sc_off["n_prompts"]} = {sc_off["identical_rate"] * 100:.0f}%',
      f'{sc_off["token_for_token_match_rate"] * 100:.2f}%'],
     ["speculation on vs speculation off",
      f'{on_off["identical_outputs"]}/{on_off["n_prompts"]} = {on_off["identical_rate"] * 100:.0f}%',
      f'{on_off["token_for_token_match_rate"] * 100:.2f}%']])
print("unit:", sc_on["unit"])

| comparison | identical completions | token-for-token match rate |
|---|---|---|
| speculation on vs itself (noise floor) | 9/20 = 45% | 60.98% |
| speculation **off** vs itself (noise floor) | 6/20 = 30% | 50.18% |
| speculation on vs speculation off | 2/20 = 10% | 33.59% |

unit: positions matching / positions compared, greedy


**Reading: the nondeterminism is not caused by speculation.** Speculation off is
*also* non-deterministic, and in fact worse — 30% identical and 50.18% token
match, against speculation-on's 45% and 60.98%. Under
PP the number of accepted draft tokens per step varies with batch composition,
which changes the shape of the verification forward pass and therefore the
reduction order in the kernels; the pipeline itself supplies the rest.

The on-versus-off row (10% identical, 33.59% token match) sits **below both
noise floors**, so it cannot be read as "speculation changes 66% of tokens." A
lossless claim on this stack would have to clear a floor that the stack itself
does not clear, and **no lossless verdict is available here.** Publishing the
on-versus-off number without the two control rows beside it would be
misleading, which is why all three are in one table.

The honest label for this cell is **not measurable on this stack** — not
"failed", and not a percentage. A configuration that disagrees with itself on
39-49% of positions cannot support a token-for-token verdict either way.

### 2.10 Speculation off: what MTP is actually worth

The speculation-off boot also gives the missing baseline for the drafter's
value. Same recipe, same protocols, `--speculative-config` removed.

In [14]:
ns_p1 = receipt("nospec", "p1.json")
ns_dc = receipt("nospec", "decode_c1.json")
k3p1 = p1["k3"]
k3dc = receipt("k3", "decode_c1.json")

rows = []
for w in WORKLOADS:
    a, b = k3p1["workloads"][w], ns_p1["workloads"][w]
    dega = any(r.get("degenerate") for r in a["reps"])
    rows.append([w,
                 f'{a["median_tok_s"]:.2f}' + (" *(deg)*" if dega else ""),
                 f'{b["median_tok_s"]:.2f}',
                 f'{a["median_tok_s"] / b["median_tok_s"]:.2f}x'])

k3reps = [r["decode_tok_s"] for r in k3dc["reps"]]
nsreps = [r["decode_tok_s"] for r in ns_dc["reps"]]
rows.append(["**P2 median (5 reps)**", f"**{st.median(k3reps):.2f}**",
             f"**{st.median(nsreps):.2f}**",
             f"**{st.median(k3reps) / st.median(nsreps):.2f}x**"])

render_table(["workload (median, tok/s)", "MTP k=3", "speculation off", "uplift"], rows)

| workload (median, tok/s) | MTP k=3 | speculation off | uplift |
|---|---|---|---|
| counting | 75.69 | 42.88 | 1.77x |
| json | 87.02 | 42.17 | 2.06x |
| code | 58.37 *(deg)* | 42.83 | 1.36x |
| math | 87.55 | 42.94 | 2.04x |
| prose | 60.54 | 42.91 | 1.41x |
| repetition | 80.21 | 42.88 | 1.87x |
| **P2 median (5 reps)** | **67.91** | **41.95** | **1.62x** |

Speculation off sits flat at roughly 42-43 tok/s on every workload — the
drafter is what creates the spread between workloads, because acceptance is
what varies.

**The uplift depends on how the P2 reps are aggregated, so all three readings
are given rather than one.** The speculation-off arm is so tight (41.71-42.09
tok/s across five reps) that the whole spread comes from which k=3 number is
used.

In [15]:
k3reps = [r["decode_tok_s"] for r in receipt("k3", "decode_c1.json")["reps"]]
nsreps = [r["decode_tok_s"] for r in receipt("nospec", "decode_c1.json")["reps"]]

render_table(
    ["aggregation of the 5 P2 reps", "MTP k=3", "speculation off", "uplift"],
    [["**median of all 5 reps** (the protocol stated in section 1)",
      f"**{st.median(k3reps):.2f}**", f"**{st.median(nsreps):.2f}**",
      f"**{st.median(k3reps) / st.median(nsreps):.2f}x**"],
     ["median of the 4 warm reps (first rep dropped as cold)",
      f"{st.median(k3reps[1:]):.2f}", f"{st.median(nsreps[1:]):.2f}",
      f"{st.median(k3reps[1:]) / st.median(nsreps[1:]):.2f}x"],
     ["peak warm rep",
      f"{max(k3reps):.2f}", f"{max(nsreps):.2f}",
      f"{max(k3reps) / max(nsreps):.2f}x"]])

print("k=3 reps       :", k3reps)
print("speculation off:", nsreps)

| aggregation of the 5 P2 reps | MTP k=3 | speculation off | uplift |
|---|---|---|---|
| **median of all 5 reps** (the protocol stated in section 1) | **67.91** | **41.95** | **1.62x** |
| median of the 4 warm reps (first rep dropped as cold) | 74.28 | 42.02 | 1.77x |
| peak warm rep | 92.58 | 42.09 | 2.20x |

k=3 reps       : [56.32, 92.58, 80.66, 66.37, 67.91]
speculation off: [41.71, 42.09, 42.09, 41.95, 41.89]


**The figure this notebook stands behind is 1.62x**, because that is the
uplift under the protocol declared in section 1 — median of five repetitions,
cold rep included, applied identically to both arms. A 1.92x figure is
obtainable from the same receipts, but only by aggregating the two arms
differently from each other, so it is not used here.

### 2.11 Negative cell: the community block drafter on this checkpoint

The public DFlash2 drafter (`incoai/GLM-5.3-Flash-DFlash2`, cc-by-nc-nd-4.0,
measurement only) is a large win on the upstream author's NVFP4 checkpoint. On
**our AWQ W4A16 checkpoint it is a net loss**, and this is the single biggest
reason the upstream headline number does not reproduce here — not the card
count, and not PP4.

Read the fast-looking cells with the guard flags: counting and json trip the
repeat guard, and the code completion opens with broken, repeated think tags.
Code and prose — the two clean workloads — collapse to roughly half of MTP k=3.

In [16]:
d7 = receipt("awq-dflash7", "p1.json")
k3 = p1["k3"]
rows = []
for w in WORKLOADS:
    a, b = k3["workloads"][w], d7["workloads"][w]
    dega = any(r.get("degenerate") for r in a["reps"])
    degb = any(r.get("degenerate") for r in b["reps"])
    rows.append([w,
                 f'{a["median_tok_s"]:.2f}' + (" *(deg)*" if dega else ""),
                 f'{b["median_tok_s"]:.2f}' + (" *(deg)*" if degb else "")])

dr, dt, ac, rate, alen = accept("awq-dflash7")
render_table(["workload (median of 3, tok/s)", "MTP k=3 (recipe of record)",
              "DFlash2 k=7 on AWQ W4A16"], rows)
render_table(["metric", "MTP k=3", "DFlash2 k=7 on AWQ"],
             [["draft acceptance rate", "65.4%", f"{rate * 100:.1f}%"],
              ["mean accepted length", "2.96", f"{alen:.2f}"],
              ["KV pool at 393,216 max len", "1,194,627 tokens (3.04x)",
               "523,657 tokens (1.33x)"],
              ["clean text on the code workload", "yes",
               "no — repeated, broken think tags"]])

| workload (median of 3, tok/s) | MTP k=3 (recipe of record) | DFlash2 k=7 on AWQ W4A16 |
|---|---|---|
| counting | 75.69 | 129.71 |
| json | 87.02 | 136.54 *(deg)* |
| code | 58.37 *(deg)* | 36.21 |
| math | 87.55 | 110.81 |
| prose | 60.54 | 32.45 |
| repetition | 80.21 | 109.55 *(deg)* |

| metric | MTP k=3 | DFlash2 k=7 on AWQ |
|---|---|---|
| draft acceptance rate | 65.4% | 41.6% |
| mean accepted length | 2.96 | 3.91 |
| KV pool at 393,216 max len | 1,194,627 tokens (3.04x) | 523,657 tokens (1.33x) |
| clean text on the code workload | yes | no — repeated, broken think tags |

### 2.12 Cells still open

Every cell the minimum coverage matrix asks for that has no receipt yet, with
its reason. These are re-run in place as receipts land.

In [17]:
render_table(
    ["cell", "status", "note"],
    [["Lossless verdict", "**no verdict available**",
      "greedy is not reproducible on this stack with or without speculation, so no comparison can clear a lossless bar — see section 2.9"],
     ["Long-context retrieval and tool use quality buckets",
      "untested (pending)", "the battery covered reasoning/math, coding and structured output only"],
     ["Sustained power (integrated energy)", "untested (pending)",
      "only point-in-time samples after each stability round were recorded"],
     ["NVFP4 checkpoint, any topology", "**not viable on this pool** (measured)",
      "two attempts, same failure: see section 2.13"],
     ["Thinking on vs off", "**negative result**",
      "not switchable on this checkpoint — the probe got no `reasoning_content` under any key or default; the two sweep arms are the same configuration"],
     ["258k-token context point", "untested",
      "the request exceeded the 393,216-token server limit after prompt calibration; not retried"],
     ["A100-class comparison", "untested", "no A100 in the pool"],
     ["Accepted tokens per pass vs context length", "untested (pending)",
      "no SpecDecoding metrics line fell inside a sample window in the sweep run"]])

display(Markdown(
    "**A node fault interrupted this lane mid-way.** All four accelerators "
    "raised a fault requiring an OS-level recovery action and the driver could "
    "no longer initialise anywhere on the node, while the management interface "
    "kept reporting the cards as healthy. The node was recovered and the "
    "remaining cells were measured afterwards; the recipe's sixth boot, at "
    "873 s, is the post-recovery one. The failure mode and the lesson learned "
    "from it are in the guide page's troubleshooting section."))

| cell | status | note |
|---|---|---|
| Lossless verdict | **no verdict available** | greedy is not reproducible on this stack with or without speculation, so no comparison can clear a lossless bar — see section 2.9 |
| Long-context retrieval and tool use quality buckets | untested (pending) | the battery covered reasoning/math, coding and structured output only |
| Sustained power (integrated energy) | untested (pending) | only point-in-time samples after each stability round were recorded |
| NVFP4 checkpoint, any topology | **not viable on this pool** (measured) | two attempts, same failure: see section 2.13 |
| Thinking on vs off | **negative result** | not switchable on this checkpoint — the probe got no `reasoning_content` under any key or default; the two sweep arms are the same configuration |
| 258k-token context point | untested | the request exceeded the 393,216-token server limit after prompt calibration; not retried |
| A100-class comparison | untested | no A100 in the pool |
| Accepted tokens per pass vs context length | untested (pending) | no SpecDecoding metrics line fell inside a sample window in the sweep run |

**A node fault interrupted this lane mid-way.** All four accelerators raised a fault requiring an OS-level recovery action and the driver could no longer initialise anywhere on the node, while the management interface kept reporting the cards as healthy. The node was recovered and the remaining cells were measured afterwards; the recipe's sixth boot, at 873 s, is the post-recovery one. The failure mode and the lesson learned from it are in the guide page's troubleshooting section.

### 2.13 NVFP4 is not viable on this hardware

`LibertAIDAI/GLM-5.3-Flash-NVFP4` @ `caca4e6a4ebb` (MIT) was attempted twice and
failed both times with the same signature. This is a **measured negative
result**, not a pending cell and not a tuning gap.

| Attempt | Settings | Outcome |
|---|---|---|
| 1 | util 0.90, max-model-len 393,216, partition 14,12,12,7 | out of memory in the mixture-of-experts kernel-format conversion |
| 2 | util 0.85, max-model-len 131,072, partition rebalanced to 11,12,12,10 | identical failure, same place |

The conversion asked for 7,247,757,312 bytes with 3.78 GiB free on one card and
17 MB free on another.

**Why no setting fixes it.** SM80 has no native FP4, so the checkpoint is
widened on load: about **60 GiB resident per card** against roughly 45 GiB of
NVFP4 weight share on disk. The conversion buffer is allocated *during weight
load*, before the KV budget is ever computed, so neither memory utilisation nor
context length can move it. This is a hardware-generation limit on a 4x64 GiB
SM80 pool, not a configuration problem.

**Consequence for the drafter question.** The decisive experiment — the
community block drafter on the checkpoint it was actually tuned against —
cannot be run on this hardware at all. The explanation offered in section 2.11,
that DFlash2 underperforms here because it is mismatched to our AWQ weights,
therefore remains a well-supported hypothesis rather than a demonstrated
conclusion, and this pool cannot settle it.

## 3. Reproduce

**Hardware.** 4x NVIDIA CMP 170HX (SM80, 64 GiB HBM2e each), no NVLink, no P2P
over PCIe, 180 W per-card power limit, forced airflow. See
[docs/INSTALLATION.md](../docs/INSTALLATION.md) and [docs/QC.md](../docs/QC.md)
before the first run, and the fifteen-item preflight checklist in
[docs/OPERATOR-LESSONS.md](../docs/OPERATOR-LESSONS.md).

Renting instead of owning: `scripts/rental/` carries an onstart script, an offer
finder, per-topology launchers, and a run queue for a rented multi-card box.

### Pull the image

```bash
docker pull ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905

# exact reproducibility: pull by digest instead of tag
docker pull ghcr.io/pixelml/club-170hx@sha256:62f612b49614523e6a46e1493d35d3efd1f363917129d38cc923a31053693bfb
```

### Download and verify the weights

```bash
pip install -U huggingface_hub
hf download wtdcode/GLM-5.3-Flash-AWQ-W4A16 \
  --revision abd7b07719111f137e1de8a0c1b7e01c11b74d1a \
  --local-dir <weights>
```

Expect 24 files totalling 190,843,146,533 bytes, 0 missing or mismatched. Stage
the checkpoint on local NVMe: over a network mount the shard load dominates boot
time and can turn a 17-minute boot into an hour.

### Launch

The recipe of record is `recipes/glm53-flash-4x170hx-pp4.sh` in the runtime
source branch. Its shape:

```bash
docker run -d --name <container> --gpus '"device=0,1,2,3"' \
  --shm-size 16g --ipc=host -p 127.0.0.1:<port>:8000 \
  -e HF_HUB_OFFLINE=1 \
  -e VLLM_PP_LAYER_PARTITION=14,12,12,7 \
  -e VLLM_WORKER_MULTIPROC_METHOD=spawn \
  -e TORCH_CUDA_ARCH_LIST=8.0 \
  -e VLLM_PP_MAX_DECODE_REQS_PER_BATCH=2 \
  -e VLLM_GLM5N_SIDECAR_BLOCK_SIZE=256 \
  -v <weights>:/weights:ro \
  ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905 \
  --model /weights --served-model-name GLM-5.3-Flash \
  --pipeline-parallel-size 4 \
  --max-model-len 393216 \
  --gpu-memory-utilization 0.90 \
  --no-enable-prefix-caching \
  --max-num-seqs 8 --max-num-batched-tokens 4096 \
  --speculative-config '{"method":"mtp","num_speculative_tokens":3}' \
  --limit-mm-per-prompt '{"image":0,"video":0}'
```

Expected boot: about 17 minutes (1,029 s measured) from locally staged weights,
of which 320 s is engine init.

**Why each non-obvious flag is there:**

| Flag | Reason |
|---|---|
| `VLLM_PP_LAYER_PARTITION=14,12,12,7` | 45 hidden layers; 11 sparse-MLA layers sit at index 3, 7, ... 43. This split gives sparse counts 3/3/3/2 and leaves the last stage room for `lm_head` and the drafter. Even splits crash on the first request. |
| `--max-model-len 393216` | the largest length whose KV pool fits at this card count; the upstream 1,048,576 fails at boot with an explicit KV-cache size error |
| `--no-enable-prefix-caching` | the recipe measures uncached behaviour; leave it off to reproduce the numbers here (it is also why warm TTFT equals cold TTFT throughout) |
| `--gpu-memory-utilization 0.90` | 0.92 leaves too little headroom for the drafter's sidecar at this partition |
| `--limit-mm-per-prompt image:0,video:0` | text-only workaround: one card position drops off the bus under the multimodal profiling power transient on this node. Skipping that profiling stage makes text serving stable. This is a node workaround, not part of a recipe on healthy hardware, and it leaves the vision path unmeasured. |

### First request

```bash
curl http://127.0.0.1:<port>/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"GLM-5.3-Flash",
       "messages":[{"role":"user","content":"What is the capital of France? Answer with just the city name."}],
       "temperature":0,"max_tokens":128}'
```

Use `max_tokens >= 128`: this model reasons before answering and a 32-token
budget returns an empty answer.

### Bench

```bash
# P1 — the upstream author's harness, temperature 0, 512 output tokens, 3 reps
python3 scripts/bench.py --url http://127.0.0.1:<port>/v1 --model GLM-5.3-Flash

# P2 — this club's protocol, temperature 0.7 + ignore_eos, 512 output tokens, 5 reps
python3 bench_glm53.py --url http://127.0.0.1:<port>/v1 --model GLM-5.3-Flash
```

Regenerate the chart from the committed receipt, no GPU needed:

```bash
python3 assets/charts/2026-09-05-glm-5.3-flash-pp4-context-sweep.py
```

### Attribution

| Source | License | What was taken |
|---|---|---|
| [promisezackr/glm53-flash-170hx-pp8](https://github.com/promisezackr/glm53-flash-170hx-pp8) | Apache-2.0 | The pipeline-parallel patch set, 24 patches, applied verbatim over `vllm/vllm-openai:glm53-flash` @ `487ecf187` with per-patch attribution trailers. The KV-balancing rule behind the layer partition is his; the 4-stage split is our adaptation of it. His `scripts/bench.py` is the P1 protocol and its repeat guard, used unmodified. His published throughput figures are **community-reported** and are never mixed into our measured tables. |
| [wtdcode/GLM-5.3-Flash-AWQ-W4A16](https://huggingface.co/wtdcode/GLM-5.3-Flash-AWQ-W4A16) | per the model card | The AWQ W4A16 checkpoint, used as published at revision `abd7b07719111f137e1de8a0c1b7e01c11b74d1a`. Third-party verified; not re-quantized, not mirrored. |
| [incoai/GLM-5.3-Flash-DFlash2](https://huggingface.co/incoai/GLM-5.3-Flash-DFlash2) | cc-by-nc-nd-4.0 | The block drafter, downloaded for measurement only (section 2.7). Not redistributed. |

## 4. Appendix

<details>
<summary>Superseded TP4 record, pre-port negative results, the link fault, and limitations (click to expand)</summary>

### The superseded TP4 record (2026-09-03)

TP4 with MTP k=3 measured **60.4 tok/s** c=1 median (peak 77.9) and 37.0 tok/s
aggregate at c=8 on the same checkpoint and card count. It is kept here as
history, not in any current table, because TP4 is link-bound on this fabric:
every all-reduce crosses PCIe with no NVLink and no P2P, so the topology runs at
the width of its worst rank. When one card's link retrained from x8 to x1, the
identical TP4 recipe fell from **70.5 to 14.4 tok/s** under the identical
protocol, while PP4 on the same degraded box held **60.8 tok/s** — roughly 4.2x
more link-tolerant. PP moves about one hidden state per decode step; TP moves an
all-reduce per layer.

This is also the reason a code change must never be accepted or rejected on a
decode number measured across that link boundary. A four-fold regression that
looked like a code regression for two sessions was the box.

### Pre-port PP4 negative results

Before the community patch set was ported, PP4 on the stock build was
functionally correct but structurally slow, and speculative decoding under PP
was broken:

| Configuration | c=1 decode | Text | Reading |
|---|---|---|---|
| PP4 + MTP k=5, stock build | 3.35 tok/s | degenerate — `"such such such"` | MTP-under-PP was miscompiled; the draft head loaded random-init |
| PP4, speculation off, stock build | 6.11 tok/s median of 3 | clean, 3/3 facts correct | removing speculation fixes the text but not the throughput; base PP4 decode collapse is independent of MTP |
| PP4 + MTP k=3, ported patch set | 60.8 tok/s (P2) | clean | the port fixes both |

Two separate findings, kept separate: the degeneration was an
MTP-under-PP artifact, and the 6-to-9 tok/s collapse was the base pipeline
hand-off on the stock build. The ported patch set resolves both.

### Block-drafter attempts that never reached a measurement

| Attempt | Outcome |
|---|---|
| PP4 + DFlash2, stock build | Refused at init: the block drafter's auxiliary hidden-state layers (5, 14, 24, 33, 42 of 45) cannot all live on the last pipeline stage under any genuine 4-way split, and the aux relay resolves layer names without forwarding hidden states across stages. Needs an upstream cross-stage relay. |
| TP4 + DFlash2, stock build | Refused at KV-cache setup, both with and without prefix caching: page size is not divisible by the maximum page size and cannot be padded for MLA indexer layers. Prefix caching is refuted as the trigger. Needs upstream padding support for MLA layers. |
| PP4 + DFlash2 k=7, ported patch set, AWQ checkpoint | Boots and serves, but is a net loss — see section 2.7. |

### Drafter research lane

A separate lane is training a checkpoint-matched drafter, on the hypothesis that
acceptance, not depth, is the lever on this box. **In progress, no publishable
numbers yet.**

### Limitations

- Every aggregate, prefill and TTFT number in this notebook was measured with
  one card's PCIe link trained to Gen1 x1 against a slot ceiling of x8. They are
  lower bounds. c=1 decode is link-insensitive under PP4 and carries the
  headline.
- The vision path is unmeasured: the recipe disables multimodal profiling as a
  node workaround.
- No BF16 reference can be loaded on this pool, so quantization parity cannot be
  measured here. The quality section of the guide page says so rather than
  implying parity.
- The thinking switch is not switchable on this checkpoint, so the two sweep
  arms are one configuration measured twice, not an A/B.
- The NVFP4 checkpoint is not claim-ready: it was attempted and ran out of
  memory during the mixture-of-experts kernel-format conversion, so the
  decisive drafter-versus-checkpoint test is still open.
- No lossless verdict is available: greedy output on this stack is not
  reproducible with speculation on *or* off, so no token-for-token comparison
  can clear a lossless bar.
- The quality battery covers reasoning/math, coding and structured output.
  Long-context retrieval and tool use are untested, and no BF16 reference
  exists on this pool, so no parity claim is made.
- Power was sampled point-in-time after each stability round, not integrated,
  so sustained energy draw is untested.

### Evidence

Raw receipts for every table above are committed under
`results/2026-09-05-glm-5.3-flash-4card-pp4-vllm/receipts/`, and mirrored with
the run manifest and redacted commands in
[PixelML/GLM-5.3-Flash-CMP-170HX](https://github.com/PixelML/GLM-5.3-Flash-CMP-170HX).

</details>